# Korpus- und Kandidatensuche für Pakistan–Islamabad

Dieses Notebook untersucht den exakten Trainingssplit des Basismodells nach
möglichen Trainingsbelegen für die Relation
`capital(Pakistan, Islamabad)`.

Die automatische Suche erzeugt ausschließlich Kandidatenfenster. Die
anschließende inhaltliche Klassifikation der Fenster wurde manuell durchgeführt
und ist daher bewusst nicht Bestandteil dieses Notebooks.


## 1. Trainingssplit laden

Verwendet wird der beim Basismodelltraining gespeicherte exakte Trainingstokenstrom.
Kontextlänge und Stride werden direkt aus dem Cache übernommen.


In [ ]:
from pathlib import Path

import torch
import tiktoken
import pandas as pd


TRAIN_CACHE_PATH = Path(
    "/content/drive/MyDrive/simplewiki_train_tokens_exact.pt"
)

cache = torch.load(
    TRAIN_CACHE_PATH,
    map_location="cpu",
)

print(cache.keys())

TRAIN_TOKENS = cache["train_tokens"]
CONTEXT_LENGTH = cache["context_length"]
STRIDE = cache["stride"]

TOKENIZER = tiktoken.get_encoding("gpt2")

print(f"Trainingstoken:  {len(TRAIN_TOKENS):,}")
print(f"Kontextlänge:    {CONTEXT_LENGTH:,}")
print(f"Stride:          {STRIDE:,}")

NUM_WINDOWS = 1 + (
    len(TRAIN_TOKENS)
    - CONTEXT_LENGTH
    - 1
) // STRIDE

print(f"Trainingsfenster: {NUM_WINDOWS:,}")

## 2. Islamabad-Vorkommen und Relationskandidaten suchen

Für jedes 1024-Token-Fenster mit einer Nennung von `Islamabad` wird geprüft,
ob zusätzlich `capital` und `Pakistan` im selben Fenster vorkommen. Für die
manuelle Sichtung wird ein Textausschnitt um die Fundstelle gespeichert.


In [ ]:
import re


SEARCH_PATTERN = re.compile(
    r"\bIslamabad\b",
    flags=re.IGNORECASE,
)


def make_snippet(
    text: str,
    match_start: int,
    match_end: int,
    radius: int = 300,
) -> str:
    start = max(0, match_start - radius)
    end = min(len(text), match_end + radius)

    return (
        text[start:end]
        .replace("\n", " ")
        .strip()
    )


rows = []

for window_id in range(NUM_WINDOWS):
    token_start = window_id * STRIDE
    token_end = token_start + CONTEXT_LENGTH + 1

    # 1.024 Input-Token plus das letzte Target-Token.
    window_tokens = TRAIN_TOKENS[
        token_start:token_end
    ]

    window_text = TOKENIZER.decode(
        window_tokens.tolist()
    )

    matches = list(
        SEARCH_PATTERN.finditer(window_text)
    )

    if not matches:
        continue

    text_lower = window_text.casefold()

    for occurrence_index, match in enumerate(matches):
        rows.append({
            "window_id": window_id,
            "token_start": token_start,
            "token_end": token_end,
            "occurrence_index": occurrence_index,
            "contains_islamabad": True,
            "contains_capital": (
                "capital" in text_lower
            ),
            "contains_pakistan": (
                "pakistan" in text_lower
            ),
            "strict_candidate": (
                "capital" in text_lower
                and "pakistan" in text_lower
            ),
            "snippet": make_snippet(
                window_text,
                match.start(),
                match.end(),
            ),
            "manual_label": None,
            "manual_notes": None,
        })


ISLAMABAD_OCCURRENCES = pd.DataFrame(rows)

print(
    "Islamabad-Nennungen in Trainingsfenstern:",
    len(ISLAMABAD_OCCURRENCES),
)

print(
    "Unterschiedliche Islamabad-Fenster:",
    ISLAMABAD_OCCURRENCES["window_id"].nunique(),
)

## 3. Strenge Kandidatenfenster

Als strenge Kandidaten gelten Fenster, in denen `Islamabad`, `capital` und
`Pakistan` gemeinsam vorkommen.


In [ ]:
STRICT_CANDIDATES = (
    ISLAMABAD_OCCURRENCES[
        ISLAMABAD_OCCURRENCES[
            "strict_candidate"
        ]
    ]
    .drop_duplicates("window_id")
    .sort_values("window_id")
    .reset_index(drop=True)
)

print(
    "Strenge Relationskandidaten:",
    len(STRICT_CANDIDATES),
)

display(
    STRICT_CANDIDATES[
        [
            "window_id",
            "snippet",
            "manual_label",
            "manual_notes",
        ]
    ]
)

## 4. Breitere Kandidatensicht

Zusätzlich werden Fenster betrachtet, in denen `Islamabad` und `capital`
gemeinsam vorkommen. Im verwendeten Trainingssplit ergeben breite und strenge
Suche dieselben 35 Fenster.


In [ ]:
BROAD_CANDIDATES = (
    ISLAMABAD_OCCURRENCES[
        ISLAMABAD_OCCURRENCES[
            "contains_capital"
        ]
    ]
    .drop_duplicates("window_id")
    .sort_values("window_id")
    .reset_index(drop=True)
)

print(
    "Breite Relationskandidaten:",
    len(BROAD_CANDIDATES),
)

display(
    BROAD_CANDIDATES[
        [
            "window_id",
            "contains_pakistan",
            "snippet",
            "manual_label",
            "manual_notes",
        ]
    ]
)

## 5. Zusammenfassung


In [ ]:
SCREENING_SUMMARY = {
    "fact_id": "pakistan_capital_islamabad",
    "object_mentions_in_training_windows": int(
        len(ISLAMABAD_OCCURRENCES)
    ),
    "distinct_object_windows": int(
        ISLAMABAD_OCCURRENCES[
            "window_id"
        ].nunique()
    ),
    "windows_with_capital": int(
        BROAD_CANDIDATES[
            "window_id"
        ].nunique()
    ),
    "windows_with_capital_and_pakistan": int(
        STRICT_CANDIDATES[
            "window_id"
        ].nunique()
    ),
}

SCREENING_SUMMARY

## 6. Kandidaten für die manuelle Klassifikation exportieren

Die CSV enthält pro Kandidatenfenster einen Snippet sowie leere Spalten für
manuelle Labels und Notizen.


In [ ]:
ANNOTATION_PATH = Path(
    "/content/drive/MyDrive/"
    "pakistan_islamabad_evidence_candidates.csv"
)

BROAD_CANDIDATES.to_csv(
    ANNOTATION_PATH,
    index=False,
)

print(f"Gespeichert: {ANNOTATION_PATH}")

## 7. Manuelle Evidenzklassifikation

Die exportierten 35 Kandidatenfenster wurden anschließend händisch anhand ihres
vollständigen Kontexts klassifiziert. Dieser Schritt wurde nicht automatisiert
und ist deshalb in diesem Notebook nicht als Codezelle abgebildet.
